# SWAG: Stochastic Weight Averaging - Gaussian

This notebook demonstrates SWAG for single-training-run Bayesian uncertainty estimation.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.models import MLP
from deepuq.methods.swag import SWAGCollector, SWAGWrapper, MultiSWAG

## Generate Toy Data

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# Training data: 1D sine wave with noise
x_train = torch.FloatTensor(100, 1).uniform_(-3, 3)
y_train = torch.sin(x_train) + 0.1 * torch.randn_like(x_train)

# Test data (includes OOD regions)
x_test = torch.linspace(-5, 5, 200).unsqueeze(1)

plt.scatter(x_train.numpy(), y_train.numpy(), alpha=0.5, label='Training data')
plt.plot(x_test.numpy(), np.sin(x_test.numpy()), 'r--', label='True function')
plt.axvspan(-5, -3, alpha=0.1, color='red', label='OOD region')
plt.axvspan(3, 5, alpha=0.1, color='red')
plt.legend()
plt.title('Toy Regression Dataset')
plt.show()

## Train Base Model

In [ ]:
model = MLP(input_dim=1, hidden_dims=[64, 64], output_dim=1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

# Train for 200 epochs
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    pred = model(x_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

## Collect SWAG Statistics

In [ ]:
collector = SWAGCollector(model, max_rank=10)

# Continue training with constant LR, collecting weights each epoch
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

model.train()
for epoch in range(50):
    optimizer.zero_grad()
    pred = model(x_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()
    collector.collect(model)

collector.finalize()
print(f"Collected {collector.n_collected} weight snapshots")
print(f"Deviation matrix shape: {collector.deviation_matrix.shape}")

## Make Predictions with Uncertainty

In [ ]:
swag_model = SWAGWrapper(model, collector)
result = swag_model.predict_uq(x_test, n_samples=30)

mean = result.mean.numpy()
std = result.epistemic_var.sqrt().numpy()
print(f"Prediction shape: {mean.shape}")
print(f"Method: {result.metadata['method']}")

## Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_np = x_test.numpy().flatten()
mean_flat = mean.flatten()
std_flat = std.flatten()

ax.fill_between(x_np, mean_flat - 2*std_flat, mean_flat + 2*std_flat,
                alpha=0.3, color='blue', label='±2 std (epistemic)')
ax.plot(x_np, mean_flat, 'b-', label='SWAG mean')
ax.scatter(x_train.numpy(), y_train.numpy(), c='black', s=10, alpha=0.5, label='Training data')
ax.plot(x_np, np.sin(x_np), 'r--', label='True function')

# Highlight OOD regions
ax.axvspan(-5, -3, alpha=0.1, color='red', label='OOD region')
ax.axvspan(3, 5, alpha=0.1, color='red')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('SWAG Uncertainty Estimation')
ax.legend()
plt.tight_layout()
plt.show()

## MultiSWAG

In [ ]:
# Train 3 SWAG models from different initializations
swag_wrappers = []

for i in range(3):
    torch.manual_seed(i * 100)
    m = MLP(input_dim=1, hidden_dims=[64, 64], output_dim=1)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    
    # Pre-train
    m.train()
    for epoch in range(200):
        opt.zero_grad()
        loss = loss_fn(m(x_train), y_train)
        loss.backward()
        opt.step()
    
    # Collect SWAG statistics
    coll = SWAGCollector(m, max_rank=10)
    opt = torch.optim.SGD(m.parameters(), lr=1e-3)
    for epoch in range(50):
        opt.zero_grad()
        loss = loss_fn(m(x_train), y_train)
        loss.backward()
        opt.step()
        coll.collect(m)
    coll.finalize()
    
    swag_wrappers.append(SWAGWrapper(m, coll))
    print(f"Model {i+1} trained and SWAG collected")

# Combine with MultiSWAG
multi_swag = MultiSWAG(swag_wrappers)
multi_result = multi_swag.predict_uq(x_test, n_samples_per_model=10)

# Compare single SWAG vs MultiSWAG
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, title in [(axes[0], result, 'Single SWAG'), (axes[1], multi_result, 'MultiSWAG (3 models)')]:
    m_np = res.mean.numpy().flatten()
    s_np = res.epistemic_var.sqrt().numpy().flatten()
    ax.fill_between(x_np, m_np - 2*s_np, m_np + 2*s_np, alpha=0.3, color='blue')
    ax.plot(x_np, m_np, 'b-', label='Mean')
    ax.scatter(x_train.numpy(), y_train.numpy(), c='black', s=10, alpha=0.5)
    ax.plot(x_np, np.sin(x_np), 'r--', label='True')
    ax.axvspan(-5, -3, alpha=0.1, color='red')
    ax.axvspan(3, 5, alpha=0.1, color='red')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()
print(f"MultiSWAG metadata: {multi_result.metadata}")